In [19]:
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. Load and prepare data
df = pd.read_csv("garments_worker_productivity.csv")
df.columns = df.columns.str.strip()

y_reg = df['actual_productivity']
df['Meets_Target'] = (df['actual_productivity'] >= df['targeted_productivity']).astype(int)
y_clf = df['Meets_Target']

X = df.drop(columns=['actual_productivity', 'Meets_Target', 'date'])

num_features = X.select_dtypes(include=['float64', 'int64']).columns.tolist()
cat_features = X.select_dtypes(include=['object']).columns.tolist()

# 2. Preprocessing Pipeline
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers = [
        ('num', num_transformer, num_features),
        ('cat', cat_transformer, cat_features)
    ])

X_processed = preprocessor.fit_transform(X)

X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X_processed, y_reg, y_clf, test_size=0.2, random_state=42
)

print("--- Part A: Scikit-Learn Implementation ---\n")

# 3. Linear Regression (Scikit-Learn)
lin_reg = LinearRegression()

start_train_reg = time.time()
lin_reg.fit(X_train, y_reg_train)
train_time_reg_sk = time.time() - start_train_reg

start_pred_reg = time.time()
y_reg_pred = lin_reg.predict(X_test)
pred_time_reg_sk = time.time() - start_pred_reg

# Updated variables to match Part C
mae_sk = mean_absolute_error(y_reg_test, y_reg_pred)
rmse_sk = np.sqrt(mean_squared_error(y_reg_test, y_reg_pred))
r2_sk = r2_score(y_reg_test, y_reg_pred)

print("Linear Regression Results:")
print(f"Training Time:   {train_time_reg_sk:.6f} seconds")
print(f"Prediction Time: {pred_time_reg_sk:.6f} seconds")
print(f"MAE:  {mae_sk:.4f}")
print(f"RMSE: {rmse_sk:.4f}")
print(f"R2:   {r2_sk:.4f}\n")

# 4. Logistic Regression (Scikit-Learn)
log_reg = LogisticRegression(max_iter=1000)

start_train_clf = time.time()
log_reg.fit(X_train, y_clf_train)
train_time_clf_sk = time.time() - start_train_clf

start_pred_clf = time.time()
y_clf_pred = log_reg.predict(X_test)
pred_time_clf_sk = time.time() - start_pred_clf

# Updated variables to match Part C
acc_sk = accuracy_score(y_clf_test, y_clf_pred)
prec_sk = precision_score(y_clf_test, y_clf_pred)
rec_sk = recall_score(y_clf_test, y_clf_pred)
f1_sk = f1_score(y_clf_test, y_clf_pred)

print("Logistic Regression Results:")
print(f"Training Time:   {train_time_clf_sk:.6f} seconds")
print(f"Prediction Time: {pred_time_clf_sk:.6f} seconds")
print(f"Accuracy:  {acc_sk:.4f}")
print(f"Precision: {prec_sk:.4f}")
print(f"Recall:    {rec_sk:.4f}")
print(f"F1-Score:  {f1_sk:.4f}")

--- Part A: Scikit-Learn Implementation ---

Linear Regression Results:
Training Time:   0.007081 seconds
Prediction Time: 0.000507 seconds
MAE:  0.1085
RMSE: 0.1486
R2:   0.1682

Logistic Regression Results:
Training Time:   0.012647 seconds
Prediction Time: 0.000000 seconds
Accuracy:  0.7750
Precision: 0.7808
Recall:    0.9661
F1-Score:  0.8636


In [20]:
df = pd.read_csv("garments_worker_productivity.csv")
df.columns = df.columns.str.strip()

y_reg_raw = df['actual_productivity'].values
y_clf_raw = (df['actual_productivity'] >= df['targeted_productivity']).astype(int).values
X_raw = df.drop(columns=['actual_productivity', 'date'])

np.random.seed(42)
indices = np.random.permutation(len(X_raw))
test_size = int(0.2 * len(X_raw))
test_idx, train_idx = indices[:test_size], indices[test_size:]

X_train_raw = X_raw.iloc[train_idx].copy()
X_test_raw = X_raw.iloc[test_idx].copy()
y_reg_train = y_reg_raw[train_idx]
y_reg_test = y_reg_raw[test_idx]
y_clf_train = y_clf_raw[train_idx]
y_clf_test = y_clf_raw[test_idx]

num_cols = X_train_raw.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X_train_raw.select_dtypes(include=['object']).columns

# A. Missing Value Handling (Fit on train, apply to both)
medians = X_train_raw[num_cols].median()
X_train_raw[num_cols] = X_train_raw[num_cols].fillna(medians)
X_test_raw[num_cols] = X_test_raw[num_cols].fillna(medians)

X_train_raw[cat_cols] = X_train_raw[cat_cols].fillna('missing')
X_test_raw[cat_cols] = X_test_raw[cat_cols].fillna('missing')

# B. Categorical Encoding (One-Hot Encoding via pandas)
X_train_ohe = pd.get_dummies(X_train_raw, columns=cat_cols)
X_test_ohe = pd.get_dummies(X_test_raw, columns=cat_cols)
# Align columns to ensure train and test have the exact same dummy features
X_train_ohe, X_test_ohe = X_train_ohe.align(X_test_ohe, join='left', axis=1, fill_value=0)

# C. Feature Scaling (Standardization: (X - mean) / std)
means = X_train_ohe[num_cols].mean()
stds = X_train_ohe[num_cols].std(ddof=0)
stds[stds == 0] = 1 # Prevent division by zero

X_train_ohe[num_cols] = (X_train_ohe[num_cols] - means) / stds
X_test_ohe[num_cols] = (X_test_ohe[num_cols] - means) / stds

# Convert to NumPy arrays and add bias term (column of 1s) for the models
X_train_mat = X_train_ohe.to_numpy(dtype=float)
X_test_mat = X_test_ohe.to_numpy(dtype=float)

X_train_mat = np.c_[np.ones(X_train_mat.shape[0]), X_train_mat]
X_test_mat = np.c_[np.ones(X_test_mat.shape[0]), X_test_mat]

def calc_mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def calc_rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred)**2))

def calc_r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - np.mean(y_true))**2)
    return 1 - (ss_res / ss_tot)

def calc_clf_metrics(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    
    accuracy = (tp + tn) / len(y_true)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return accuracy, precision, recall, f1
# (Keep all your manual data loading, preprocessing, and metric functions exactly as they are up to the model classes)

print("--- PART B: FROM-SCRATCH IMPLEMENTATION ---\n")

class ManualLinearRegression:
    def fit(self, X, y):
        self.theta = np.linalg.pinv(X.T @ X) @ X.T @ y
        
    def predict(self, X):
        return X @ self.theta

manual_lin_reg = ManualLinearRegression()

start_train_reg = time.time()
manual_lin_reg.fit(X_train_mat, y_reg_train)
train_time_reg_man = time.time() - start_train_reg

start_pred_reg = time.time()
y_reg_pred_manual = manual_lin_reg.predict(X_test_mat)
pred_time_reg_man = time.time() - start_pred_reg

# Store metrics to match Part C
mae_man = calc_mae(y_reg_test, y_reg_pred_manual)
rmse_man = calc_rmse(y_reg_test, y_reg_pred_manual)
r2_man = calc_r2(y_reg_test, y_reg_pred_manual)

print("Manual Linear Regression Results:")
print(f"Training Time:   {train_time_reg_man:.6f} seconds")
print(f"Prediction Time: {pred_time_reg_man:.6f} seconds")
print(f"MAE:  {mae_man:.4f}")
print(f"RMSE: {rmse_man:.4f}")
print(f"R2:   {r2_man:.4f}\n")

class ManualLogisticRegression:
    def __init__(self, lr=0.1, max_iter=2000):
        self.lr = lr
        self.max_iter = max_iter
        
    def sigmoid(self, z):
        z = np.clip(z, -250, 250)
        return 1 / (1 + np.exp(-z))
        
    def fit(self, X, y):
        m, n = X.shape
        self.theta = np.zeros(n)
        for _ in range(self.max_iter):
            h = self.sigmoid(X @ self.theta)
            gradient = (1 / m) * X.T @ (h - y)
            self.theta -= self.lr * gradient
            
    def predict_proba(self, X):
        return self.sigmoid(X @ self.theta)
        
    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

manual_log_reg = ManualLogisticRegression(lr=0.1, max_iter=2000)

start_train_clf = time.time()
manual_log_reg.fit(X_train_mat, y_clf_train)
train_time_clf_man = time.time() - start_train_clf

start_pred_clf = time.time()
y_clf_pred_manual = manual_log_reg.predict(X_test_mat)
pred_time_clf_man = time.time() - start_pred_clf

# Store metrics to match Part C
acc_man, prec_man, rec_man, f1_man = calc_clf_metrics(y_clf_test, y_clf_pred_manual)

print("Manual Logistic Regression Results:")
print(f"Training Time:   {train_time_clf_man:.6f} seconds")
print(f"Prediction Time: {pred_time_clf_man:.6f} seconds")
print(f"Accuracy:  {acc_man:.4f}")
print(f"Precision: {prec_man:.4f}")
print(f"Recall:    {rec_man:.4f}")
print(f"F1-Score:  {f1_man:.4f}")

--- PART B: FROM-SCRATCH IMPLEMENTATION ---

Manual Linear Regression Results:
Training Time:   0.000000 seconds
Prediction Time: 0.001049 seconds
MAE:  0.1087
RMSE: 0.1489
R2:   0.1681

Manual Logistic Regression Results:
Training Time:   0.096396 seconds
Prediction Time: 0.000000 seconds
Accuracy:  0.7741
Precision: 0.7798
Recall:    0.9659
F1-Score:  0.8629


In [21]:
comparison_data = {
    "Model": [
        "Linear Reg (Scikit-learn)", 
        "Linear Reg (Manual)",
        "Logistic Reg (Scikit-learn)", 
        "Logistic Reg (Manual)"
    ],
    "Train Time (s)": [
        train_time_reg_sk, 
        train_time_reg_man, 
        train_time_clf_sk, 
        train_time_clf_man
    ],
    "Predict Time (s)": [
        pred_time_reg_sk, 
        pred_time_reg_man, 
        pred_time_clf_sk, 
        pred_time_clf_man
    ],
    "Metric 1": [
        f"MAE: {mae_sk:.4f}", 
        f"MAE: {mae_man:.4f}", 
        f"Acc: {acc_sk:.4f}", 
        f"Acc: {acc_man:.4f}"
    ],
    "Metric 2": [
        f"RMSE: {rmse_sk:.4f}", 
        f"RMSE: {rmse_man:.4f}", 
        f"F1: {f1_sk:.4f}", 
        f"F1: {f1_man:.4f}"
    ]
}

comp_df = pd.DataFrame(comparison_data)
print("\n--- PART C: COMPARISON TABLE ---")
print(comp_df.to_string(index=False))


--- PART C: COMPARISON TABLE ---
                      Model  Train Time (s)  Predict Time (s)    Metric 1     Metric 2
  Linear Reg (Scikit-learn)        0.007081          0.000507 MAE: 0.1085 RMSE: 0.1486
        Linear Reg (Manual)        0.000000          0.001049 MAE: 0.1087 RMSE: 0.1489
Logistic Reg (Scikit-learn)        0.012647          0.000000 Acc: 0.7750   F1: 0.8636
      Logistic Reg (Manual)        0.096396          0.000000 Acc: 0.7741   F1: 0.8629
